In [1]:
# Week12 start
# dependencies: pandas, numpy, scikit-learn, pickle
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

In [2]:
# load data and define features
projectDir = Path.cwd().resolve()
if not (projectDir / "data").exists():
    projectDir = projectDir.parent

sourceDataPath = projectDir / "data" / "processed" / "matchupDiff_week5_features.csv"
dataDf = pd.read_csv(sourceDataPath)

leakCols = ["WScore", "LScore", "WTeamID", "LTeamID", "winnerTeamId", "team1Name", "team2Name", "Unnamed: 0", "team1Id", "team2Id"]
dropCols = ["Season", "y"] + leakCols
featureCols = [col for col in dataDf.columns if col not in dropCols]

baselineDf = dataDf[dataDf["Season"] <= 2017].copy()
recentDf = dataDf[dataDf["Season"] >= 2018].copy()

print(dataDf.shape, len(featureCols))
print(baselineDf.shape, recentDf.shape)

(1353, 120) 108
(964, 120) (389, 120)


In [3]:
# train model and serialize with pickle
trainX = baselineDf[featureCols]
trainY = baselineDf["y"].astype(int)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(n_estimators=300, random_state=42))
])
model.fit(trainX, trainY)

modelPayload = {
    "model": model,
    "modelFamily": "randomForest",
    "featureCols": featureCols,
    "threshold": 0.5
}

deployModelPath = projectDir / "models" / "week12_deployed_model.pkl"
with open(deployModelPath, "wb") as f:
    pickle.dump(modelPayload, f)

with open(deployModelPath, "rb") as f:
    deployedPayload = pickle.load(f)

print(deployModelPath.name, deployedPayload["modelFamily"])

week12_deployed_model.pkl randomForest


In [4]:
# save and load monitoring data
baselinePath = projectDir / "data" / "processed" / "week12_baseline_data.csv"
recentPath = projectDir / "data" / "processed" / "week12_recent_data.csv"

baselineDf.to_csv(baselinePath, index=False)
recentDf.to_csv(recentPath, index=False)

baselineLoadedDf = pd.read_csv(baselinePath)
recentLoadedDf = pd.read_csv(recentPath)

print(baselineLoadedDf.shape, recentLoadedDf.shape)

(964, 120) (389, 120)


In [5]:
# data drift check with psi
def calculatePsi(baseSeries, recentSeries, bins=10):
    baseClean = baseSeries.dropna()
    recentClean = recentSeries.dropna()
    edges = np.unique(np.quantile(baseClean, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    baseCounts, _ = np.histogram(baseClean, bins=edges)
    recentCounts, _ = np.histogram(recentClean, bins=edges)
    basePct = np.clip(baseCounts / baseCounts.sum(), 1e-6, None)
    recentPct = np.clip(recentCounts / recentCounts.sum(), 1e-6, None)
    return float(np.sum((recentPct - basePct) * np.log(recentPct / basePct)))

monitorCols = featureCols[:12]
psiRows = []
for col in monitorCols:
    psiRows.append({"feature": col, "psi": calculatePsi(baselineLoadedDf[col], recentLoadedDf[col])})

dataDriftDf = pd.DataFrame(psiRows).sort_values("psi", ascending=False)
dataDriftDf.head(10)

,feature,psi
4,Active Coaching Length Index_diff,0.099055
8,AdjTempo_diff,0.075089
7,AdjOE_diff,0.057825
11,Adjusted Offensive Efficiency_diff,0.057714
6,AdjEM_diff,0.053583
1,team1_Seed,0.039862
3,ARate_diff,0.034447
2,team2_Seed,0.033399
10,Adjusted Defensive Efficiency Rank_diff,0.031522
0,DayNum,0.020722


In [6]:
# concept drift check from performance change
model = deployedPayload["model"]
threshold = float(deployedPayload["threshold"])

baselineX = baselineLoadedDf[featureCols]
recentX = recentLoadedDf[featureCols]
baselineY = baselineLoadedDf["y"].astype(int)
recentY = recentLoadedDf["y"].astype(int)

baselineProb = model.predict_proba(baselineX)[:, 1]
recentProb = model.predict_proba(recentX)[:, 1]
baselinePred = (baselineProb >= threshold).astype(int)
recentPred = (recentProb >= threshold).astype(int)

baselineAccuracy = float((baselinePred == baselineY).mean())
recentAccuracy = float((recentPred == recentY).mean())
accuracyDrop = baselineAccuracy - recentAccuracy

seasonRows = []
for seasonValue, seasonDf in recentLoadedDf.groupby("Season"):
    seasonProb = model.predict_proba(seasonDf[featureCols])[:, 1]
    seasonPred = (seasonProb >= threshold).astype(int)
    seasonRows.append({
        "season": int(seasonValue),
        "nGames": int(len(seasonDf)),
        "accuracy": float((seasonPred == seasonDf["y"].astype(int)).mean()),
        "positiveRate": float(seasonPred.mean())
    })

conceptDriftDf = pd.DataFrame(seasonRows).sort_values("season")
print("baselineAccuracy", round(baselineAccuracy, 4))
print("recentAccuracy", round(recentAccuracy, 4))
print("accuracyDrop", round(accuracyDrop, 4))
conceptDriftDf.tail(6)

baselineAccuracy 1.0
recentAccuracy 0.7198
accuracyDrop 0.2802


,season,nGames,accuracy,positiveRate
0,2018,66,0.727273,0.878788
1,2019,67,0.746269,0.805970
2,2021,65,0.753846,0.892308
3,2022,63,0.698413,0.841270
4,2023,61,0.737705,0.852459
5,2024,67,0.656716,0.865672


In [7]:
# monitoring plan and drift impact
monitoringPlanDf = pd.DataFrame([
    {"metric": "topFeaturePsi", "threshold": 0.20, "cadence": "weekly", "owner": "dataScience"},
    {"metric": "modelAccuracy", "threshold": "drop>0.05", "cadence": "weekly", "owner": "dataScience"},
    {"metric": "predictionPositiveRate", "threshold": "delta>0.10", "cadence": "weekly", "owner": "analytics"},
    {"metric": "retrainDecision", "threshold": "2 alerts in a row", "cadence": "monthly", "owner": "team"}
])

planPath = projectDir / "data" / "processed" / "week12_monitoring_plan.csv"
monitoringPlanDf.to_csv(planPath, index=False)
monitoringPlanLoadedDf = pd.read_csv(planPath)

maxPsi = float(dataDriftDf["psi"].max())
dataDriftImpact = bool((dataDriftDf["psi"] > 0.20).any())
conceptDriftImpact = bool(accuracyDrop > 0.05)

print("maxPsi", round(maxPsi, 4))
print("dataDriftImpact", dataDriftImpact)
print("conceptDriftImpact", conceptDriftImpact)
print("dataDriftConclusion", "drift is affecting input distributions" if dataDriftImpact else "drift is small right now")
print("conceptDriftConclusion", "performance drop suggests concept drift" if conceptDriftImpact else "no large performance drift yet")
monitoringPlanLoadedDf

maxPsi 0.0991
dataDriftImpact False
conceptDriftImpact True
dataDriftConclusion drift is small right now
conceptDriftConclusion performance drop suggests concept drift


,metric,threshold,cadence,owner
0,topFeaturePsi,0.2,weekly,dataScience
1,modelAccuracy,drop>0.05,weekly,dataScience
2,predictionPositiveRate,delta>0.10,weekly,analytics
3,retrainDecision,2 alerts in a row,monthly,team
